# 04 | Neural networks and manual backpropagation

**Question:** Can a small network learn a nonlinear decision boundary using explicitly derived gradients?

This notebook is an AI-assisted portfolio extension of ECE 657 coursework, not an original submitted assignment. Run all cells from top to bottom after installing the repository requirements.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'projects').is_dir(): root = root.parent
sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt

## 1. Model and objective
A two-input, 16-hidden-unit ReLU network predicts a binary class. With logits $z$, stable binary cross-entropy is $\\operatorname{softplus}(z)-yz$. The original coursework had a sign error in the loss; this reproduction corrects it.

In [ ]:
import jax
import jax.numpy as jnp
from projects.neural_network import initialize, logits, loss, manual_grad, experiment
import inspect
print(inspect.getsource(manual_grad))

## 2. Derive and check the gradients
With $a_1=\\operatorname{ReLU}(XW_1+b_1)$ and $z_2=a_1W_2+b_2$, the output derivative is $(\\sigma(z_2)-y)/N$. Propagate backward through the second weight matrix and ReLU mask. Compare each parameter gradient against automatic differentiation.

In [ ]:
p=initialize(); x=jnp.array([[.1,.2],[-.4,.7],[.8,-.3]]); y=jnp.array([[1.],[0.],[0.]])
manual=manual_grad(p,x,y); reference=jax.grad(loss)(p,x,y)
for key in manual:
    print(key,float(jnp.max(jnp.abs(manual[key]-reference[key]))))
    np.testing.assert_allclose(manual[key],reference[key],atol=1e-6)

## 3. Train and assess held-out data
The training loop uses the manual gradient, not autodiff. An independently held-out split estimates performance; balanced accuracy guards against the majority-class baseline.

In [ ]:
metrics,history,p,X,y=experiment(seed=42,epochs=5000)
print(metrics)
plt.plot(np.arange(len(history))*100,history); plt.xlabel('Epoch'); plt.ylabel('Binary cross-entropy'); plt.title('Training loss'); plt.show()

In [ ]:
xx,yy=np.meshgrid(np.linspace(-1,1,250),np.linspace(-1,1,250))
zz=np.asarray(logits(p,jnp.asarray(np.c_[xx.ravel(),yy.ravel()],dtype=jnp.float32))).reshape(xx.shape)
fig,ax=plt.subplots(figsize=(5,5)); ax.contourf(xx,yy,zz>0,levels=[-.5,.5,1.5],alpha=.25,cmap='coolwarm')
ax.scatter(X[:,0],X[:,1],c=y,s=8,cmap='coolwarm'); ax.add_patch(plt.Circle((0,0),.5,fill=False,linestyle='--',label='True boundary'))
ax.set(aspect='equal',xlabel='Feature 1',ylabel='Feature 2'); ax.legend(); plt.show()

## 4. Interpretation
Recorded run: 98% test accuracy, 97.6% balanced accuracy, versus an 80% majority-class baseline. Gradient discrepancies were approximately $10^{-7}$. A toy boundary verifies the implementation, but does not establish performance on real-world data.

**Practice:** Compare hidden widths and random seeds using a validation split. Explain why a single linear classifier cannot represent a circle.